In [ ]:
import pypsa
import pandas as pd
import numpy as np


In [ ]:
network = pypsa.Network()

timestamps = pd.date_range("2025-01-01 00:00", periods=24, freq="h")
network.set_snapshots(timestamps) 

In [ ]:
for i in range(1, 34):
    network.add("Bus",
                f"Bus_{i}",
                v_nom=12.66)
    
network.buses

In [ ]:
# r per length, x per length; 1 km
line_data = [
    (1, 2, 0.0922, 0.047),      (2, 3, 0.493, 0.2511),      (3, 4, 0.366, 0.1864), 
    (4, 5, 0.3811, 0.1941),     (5, 6, 0.819, 0.707),       (6, 7, 0.1872, 0.6188), 
    (7, 8, 0.7114, 0.2351),     (8, 9, 1.03, 0.74),         (9, 10, 1.044, 0.74),   
    (10, 11, 0.1966, 0.065),    (11, 12, 0.3744, 0.198),    (12, 13, 1.468, 1.155),
    (13, 14, 0.5416, 0.7129),   (14, 15, 0.591, 0.526),     (15, 16, 0.7463, 0.545),
    (16, 17, 1.289, 1.721),     (17, 18, 0.732, 0.574),     (2, 19, 0.164, 0.1565),
    (19, 20, 1.5042, 1.3554),   (20, 21, 0.4095, 0.4784),   (21, 22, 0.7089, 0.9373),   
    (3, 23, 0.4512, 0.3083),    (23, 24, 0.898, 0.7091),    (24, 25, 0.896, 0.7011), 
    (6, 26, 0.203, 0.1034),     (26, 27, 0.2842, 0.1447),   (27, 28, 1.059, 0.9337), 
    (28, 29, 0.8042, 0.7006),   (29, 30, 0.5075, 0.2585),   (30, 31, 0.9744, 0.963), 
    (31, 32, 0.3105, 0.3619),   (32, 33, 0.341, 0.5302),
]

for i, (bus0, bus1, r, x) in enumerate(line_data):
    network.add(
        "Line",
        f"Line_{i}",
        bus0=f"Bus_{bus0}",
        bus1=f"Bus_{bus1}",
        r=r,
        x=x,
        s_nom=5000,
        # type="line"
    )
network.lines

In [ ]:
price_profile = np.array([50, 45, 40, 40, 42, 48, 60, 75, 70, 65, 50, 30,
                          20, 20, 25, 45, 65, 80, 90, 100, 85, 70, 60, 55])
grid_price_series = pd.Series(price_profile, index=network.snapshots)

In [ ]:

network.add(
    "Generator",
    "Grid Connection",
    bus="Bus_1",
    p_nom=10000, #Nominal power in pw 10 MW
    p_min_pu=-1.0,
    marginal_cost=grid_price_series, #Cost in EUR/MWh
    carrier="gas" # Assuming a conventional generator
)
network.generators

In [ ]:
load_profile = [
    (100, 60), (90, 40), (120, 80), (60, 30), (60, 20),
    (200, 100), (200, 100), (60, 20), (60, 20), (45, 30),
    (60, 35), (60, 35), (120, 80), (60, 10), (60, 20),
    (60, 20), (90, 40), (90, 40), (90, 40), (90, 40),
    (90, 40), (90, 50), (420, 200), (420, 200), (60, 25),
    (60, 25), (60, 20), (120, 70), (200, 600), (150, 70),
    (210, 100), (60, 40)
]
load = np.sin(np.linspace(0, 2 * np.pi, 24)) * 0.4 + 0.6
load_series = pd.Series(100 * load, index=network.snapshots)
# print(load)
    
 
# print(load_profile)
for i in range(2, 34):
    # (p, = load[i-2]
    network.add(
        "Load",
        f"Load_bus_{i}",
        bus=f"Bus_{i}",
        p_set=load_series
    )
    # network.loads_t.p_set.(f"Load_bus_{i}") = 100 * load_profile
network.loads_t
# network.loads

In [ ]:
pv_profile = np.sin(np.linspace(-np.pi / 2, 3 * np.pi / 2, 24))
pv_profile[pv_profile < 0] = 0
pv_p = pd.Series(pv_profile, index=network.snapshots)
print(pv_profile)
for i in range(2, 34):
    network.add(
        "Generator",
        f"PV_bus_{i}",
        bus=f"Bus_{i}",
        p_nom=150, # Nominal power in kW
        p_max_pu=pv_p, # Per-unit availability
        carrier="solar",
        marginal_cost=0
    )
network.generators

In [ ]:
for i in range(2, 34):
    network.add(
        "StorageUnit",
        f"Battery_bus_{i}",
        bus=f"Bus_{i}",
        p_nom=200, # Nominal capacity in kW
        max_hours=4, # Energy capacity is p_nom * max_hours = 800 kWh
        carrier="battery",
        efficiency_store=0.9,
        efficiency_dispatch=0.9,
        standing_loss=0.01 # 1% loss per hour
    )
network.storage_units

In [ ]:
network.optimize()

In [ ]:
network.generators_t.p

In [ ]:
network.storage_units_t.p